# WC 2026 predictions

## Objective

Use the 2026 forecast dataset and the Python `PredictGame` class to predict every possible 2026 matchup. The output is a dashboard-ready JSON file with one row per unique game, win/tie probabilities, and the 10 most likely scores.

## Inputs

- `1.DataCleaning-R/Data/CSV/WC2026ForecastData.csv`
- `3.SimulationStudy-Python/Class.py`

## Outputs

- `1.DataCleaning-R/Data/JSON/WC2026Predictions.json`
- `4.Dashboard/data/wc2026_predictions.json`

## Imports

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from scipy.stats import poisson, skellam

## Paths

Find the project root so this notebook can run from either the repository root or this notebook folder.

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "3.SimulationStudy-Python" / "Class.py").exists():
            return path
    raise FileNotFoundError("Could not find WC2026Forecast project root")


PROJECT_ROOT = find_project_root(Path.cwd())
SIM_DIR = PROJECT_ROOT / "3.SimulationStudy-Python"
FORECAST_DATA_PATH = PROJECT_ROOT / "1.DataCleaning-R" / "Data" / "CSV" / "WC2026ForecastData.csv"
DATA_JSON_PATH = PROJECT_ROOT / "1.DataCleaning-R" / "Data" / "JSON" / "WC2026Predictions.json"
DASHBOARD_JSON_PATH = PROJECT_ROOT / "4.Dashboard" / "data" / "wc2026_predictions.json"

sys.path.append(str(SIM_DIR))
from Class import PredictGame

PROJECT_ROOT

## Load Forecast Data

Each row is one unique 2026 matchup. The home/away labels are only feature prefixes; the JSON exposes them as team 1 and team 2.

In [ ]:
forecast_games = pd.read_csv(FORECAST_DATA_PATH)

required_columns = [
    "match_id",
    "home_team_name",
    "away_team_name",
    "home_ELO",
    "away_ELO",
    "home_prior_world_cups_coached",
    "away_prior_world_cups_coached",
    "home_players_with_multiple_prior_wcs",
    "away_players_with_multiple_prior_wcs",
    "home_avg_age",
    "away_avg_age",
    "home_distance_from_host_km",
    "away_distance_from_host_km",
]

missing_columns = sorted(set(required_columns) - set(forecast_games.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

forecast_games[required_columns].isna().sum().sort_values(ascending=False).head(10)
forecast_games.shape

## Prediction Helpers

Use the class to calculate team lambdas, then compute stable exact probabilities from the Poisson model. Probabilities are saved on a 0-to-1 scale.

In [ ]:
def round_prob(value, digits=6):
    return round(float(value), digits)


def make_game(row):
    return PredictGame(
        row["home_team_name"],
        row["away_team_name"],
        row["home_ELO"],
        row["away_ELO"],
        row["home_prior_world_cups_coached"],
        row["away_prior_world_cups_coached"],
        row["home_players_with_multiple_prior_wcs"],
        row["away_players_with_multiple_prior_wcs"],
        row["home_avg_age"],
        row["away_avg_age"],
        row["home_distance_from_host_km"],
        row["away_distance_from_host_km"],
    )


def top_scores(lambda_team1, lambda_team2, n=10):
    max_lambda = max(lambda_team1, lambda_team2)
    max_goals = max(12, int(np.ceil(max_lambda + 8 * np.sqrt(max_lambda + 1))))

    scores = []
    for team1_goals in range(max_goals + 1):
        for team2_goals in range(max_goals + 1):
            probability = poisson.pmf(team1_goals, lambda_team1) * poisson.pmf(team2_goals, lambda_team2)
            scores.append({
                "score": f"{team1_goals}-{team2_goals}",
                "team1_goals": int(team1_goals),
                "team2_goals": int(team2_goals),
                "probability": round_prob(probability),
            })

    return sorted(scores, key=lambda score: score["probability"], reverse=True)[:n]


def predict_match(row):
    game = make_game(row)
    lambda_team1 = float(game.lambda_home())
    lambda_team2 = float(game.lambda_away())

    winteam1_prob = float(1 - skellam.cdf(0, lambda_team1, lambda_team2))
    winteam2_prob = float(skellam.cdf(-1, lambda_team1, lambda_team2))
    tie_prob = float(skellam.pmf(0, lambda_team1, lambda_team2))

    return {
        "match_id": row["match_id"],
        "team1_name": row["home_team_name"],
        "team2_name": row["away_team_name"],
        "winteam1_prob": round_prob(winteam1_prob),
        "winteam2_prob": round_prob(winteam2_prob),
        "tie_prob": round_prob(tie_prob),
        "team1_expected_goals": round_prob(lambda_team1),
        "team2_expected_goals": round_prob(lambda_team2),
        "top_scores": top_scores(lambda_team1, lambda_team2, n=10),
    }

## Generate JSON

Create one dashboard object with metadata and a `matches` array.

In [ ]:
predictions = [predict_match(row) for _, row in forecast_games.iterrows()]

payload = {
    "metadata": {
        "tournament": "WC-2026",
        "probability_scale": "0_to_1",
        "top_scores_per_match": 10,
        "match_count": len(predictions),
        "source": str(FORECAST_DATA_PATH.relative_to(PROJECT_ROOT)),
    },
    "matches": predictions,
}

payload["metadata"]

## Quick Checks

In [ ]:
prediction_checks = pd.DataFrame({
    "matches": [len(predictions)],
    "missing_team1": [sum(not match["team1_name"] for match in predictions)],
    "missing_team2": [sum(not match["team2_name"] for match in predictions)],
    "bad_probability_sums": [sum(
        abs(match["winteam1_prob"] + match["winteam2_prob"] + match["tie_prob"] - 1) > 0.00001
        for match in predictions
    )],
    "wrong_top_score_count": [sum(len(match["top_scores"]) != 10 for match in predictions)],
})

prediction_checks
predictions[:2]

## Save

In [ ]:
DATA_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
DASHBOARD_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

for output_path in [DATA_JSON_PATH, DASHBOARD_JSON_PATH]:
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)

[str(DATA_JSON_PATH.relative_to(PROJECT_ROOT)), str(DASHBOARD_JSON_PATH.relative_to(PROJECT_ROOT))]